<a href="https://colab.research.google.com/github/dakshatakamde46-creator/Dynamic-Chatbot/blob/main/Task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install datasets scikit-learn pandas streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 5.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from datasets import load_dataset

print("Loading MedQuAD dataset from Hugging Face...")

dataset = load_dataset("lavita/MedQuAD", split="train")


df = pd.DataFrame(dataset)


if "question" not in df.columns and "text" in df.columns:

    pass


df = df.dropna(subset=["question", "answer"])


df_subset = df.head(5000).reset_index(drop=True)

print(f"Dataset loaded successfully! Total QA pairs available for use: {len(df_subset)}")
print(df_subset.head(2))

Loading MedQuAD dataset from Hugging Face...


README.md:   0%|          | 0.00/2.77k [00:00<?, ?B/s]

data/train-00000-of-00001-e36383d177026d(…): reconstructing file:   0%|          |  0.00B / 10.7MB            

data/train-00000-of-00001-e36383d177026d(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/47441 [00:00<?, ? examples/s]

Dataset loaded successfully! Total QA pairs available for use: 5000
  document_id document_source  \
0     0000559             GHR   
1     0000559             GHR   

                                        document_url category  umls_cui  \
0  https://ghr.nlm.nih.gov/condition/keratoderma-...     None  C0343073   
1  https://ghr.nlm.nih.gov/condition/keratoderma-...     None  C0343073   

  umls_semantic_types umls_semantic_group synonyms question_id  \
0                T047           Disorders     KWWH   0000559-1   
1                T047           Disorders     KWWH   0000559-2   

                 question_focus question_type  \
0  keratoderma with woolly hair   information   
1  keratoderma with woolly hair     frequency   

                                            question  \
0       What is (are) keratoderma with woolly hair ?   
1  How many people are affected by keratoderma wi...   

                                              answer  
0  Keratoderma with woolly hair is 

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("Initializing TF-IDF Vectorizer and building index...")


vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))


question_vectors = vectorizer.fit_transform(df_subset["question"].values.astype(str))

print("Retrieval engine ready!")

def retrieve_best_answer(user_query, top_k=1):
    """
    Finds the most relevant question and answer from MedQuAD
    based on TF-IDF cosine similarity.
    """
    user_vec = vectorizer.transform([user_query])
    similarities = cosine_similarity(user_vec, question_vectors).flatten()


    best_indices = similarities.argsort()[::-1][:top_k]

    results = []
    for idx in best_indices:
        results.append({
            "question": df_subset.loc[idx, "question"],
            "answer": df_subset.loc[idx, "answer"],
            "score": float(similarities[idx])
        })
    return results

Initializing TF-IDF Vectorizer and building index...
Retrieval engine ready!


In [ ]:
import re


SYMPTOMS_DB = [
    "fever", "cough", "headache", "fatigue", "pain", "nausea", "vomiting",
    "dizziness", "shortness of breath", "rash", "swelling", "bleeding",
    "diarrhea", "constipation", "weight loss", "anxiety", "depression", "seizure"
]

TREATMENTS_DB = [
    "surgery", "chemotherapy", "radiation", "antibiotics", "vaccine",
    "therapy", "transplant", "medication", "insulin", "aspirin", "ibuprofen"
]

DISEASES_DB = [
    "asthma", "diabetes", "cancer", "hypertension", "arthritis", "anemia",
    "pneumonia", "bronchitis", "epilepsy", "melanoma", "leukemia", "hepatitis"
]

def extract_medical_entities(text):
    """
    Scans input text against medical dictionaries to identify
    diseases, symptoms, and treatments.
    """
    text_lower = text.lower()
    found_symptoms = [s for s in SYMPTOMS_DB if s in text_lower]
    found_treatments = [t for t in TREATMENTS_DB if t in text_lower]
    found_diseases = [d for d in DISEASES_DB if d in text_lower]

    return {
        "Symptoms": list(set(found_symptoms)),
        "Treatments": list(set(found_treatments)),
        "Diseases": list(set(found_diseases))
    }


test_query = "What are the treatments for diabetes and symptoms of fever?"
print("Entity Extraction Test:", extract_medical_entities(test_query))

Entity Extraction Test: {'Symptoms': ['fever'], 'Treatments': [], 'Diseases': ['diabetes']}


In [ ]:
import streamlit as st
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from datasets import load_dataset


st.set_page_config(
    page_title="MedQuAD Assistant",
    page_icon="🩺",
    layout="wide"
)


@st.cache_resource
def load_medquad_system():
    dataset = load_dataset("lavita/MedQuAD", split="train")
    df = pd.DataFrame(dataset).dropna(subset=["question", "answer"])
    df_subset = df.head(5000).reset_index(drop=True)

    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
    question_vectors = vectorizer.fit_transform(df_subset["question"].values.astype(str))

    return df_subset, vectorizer, question_vectors

with st.spinner("Loading Medical Knowledge Base (MedQuAD)... Please wait."):
    df_subset, vectorizer, question_vectors = load_medquad_system()

SYMPTOMS_DB = ["fever", "cough", "headache", "fatigue", "pain", "nausea", "vomiting", "dizziness", "shortness of breath", "rash", "swelling", "bleeding", "diarrhea", "constipation", "weight loss", "anxiety", "depression", "seizure"]
TREATMENTS_DB = ["surgery", "chemotherapy", "radiation", "antibiotics", "vaccine", "therapy", "transplant", "medication", "insulin", "aspirin", "ibuprofen"]
DISEASES_DB = ["asthma", "diabetes", "cancer", "hypertension", "arthritis", "anemia", "pneumonia", "bronchitis", "epilepsy", "melanoma", "leukemia", "hepatitis"]

def extract_medical_entities(text):
    text_lower = text.lower()
    return {
        "Symptoms": [s for s in SYMPTOMS_DB if s in text_lower],
        "Treatments": [t for t in TREATMENTS_DB if t in text_lower],
        "Diseases": [d for d in DISEASES_DB if d in text_lower]
    }


st.title("🩺 MedQuAD Medical Q&A Chatbot")
st.markdown("Your clinical assistant powered by the NIH MedQuAD dataset. Ask medical questions regarding conditions, symptoms, or treatments.")


st.sidebar.header(" Medical Entity Recognition")
st.sidebar.markdown("Detected entities from your active query will display here.")


user_query = st.text_input("Enter your medical question:", placeholder="e.g., What are the symptoms of diabetes?")

if user_query:

    entities = extract_medical_entities(user_query)

    with st.sidebar:
        st.markdown(f"**Diseases:** {entities['Diseases'] if entities['Diseases'] else 'None detected'}")
        st.markdown(f"**Symptoms:** {entities['Symptoms'] if entities['Symptoms'] else 'None detected'}")
        st.markdown(f"**Treatments:** {entities['Treatments'] if entities['Treatments'] else 'None detected'}")


    user_vec = vectorizer.transform([user_query])
    similarities = cosine_similarity(user_vec, question_vectors).flatten()
    best_idx = similarities.argmax()
    best_score = similarities[best_idx]

    matched_question = df_subset.loc[best_idx, "question"]
    matched_answer = df_subset.loc[best_idx, "answer"]


    st.subheader(" Answer:")
    if best_score > 0.15:
        st.success(matched_answer)
        with st.expander("View Retrieval Details"):
            st.write(f"**Matched Database Question:** {matched_question}")
            st.write(f"**Confidence Score (Cosine Similarity):** {best_score:.4f}")
    else:
        st.warning("I couldn't find a high-confidence match in the MedQuAD database for this specific phrasing. Try phrasing your question around standard medical conditions or symptoms.")

st.markdown("---")
st.caption("Disclaimer: This tool is built strictly for educational/internship evaluation purposes and does not substitute professional medical advice.")

2026-07-22 18:38:38.423 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-22 18:38:38.434 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-22 18:38:38.441 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-22 18:38:47.106 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-22 18:38:47.314 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-07-22 18:38:47.315 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when 

DeltaGenerator()

In [ ]:
import pandas as pd
from datasets import load_dataset

print("Downloading MedQuAD dataset from Hugging Face source...")

dataset = load_dataset("lavita/MedQuAD", split="train")


df = pd.DataFrame(dataset)


if "question" in df.columns and "answer" in df.columns:
    df_clean = df[["question", "answer"]].dropna()
elif "Question" in df.columns and "Answer" in df.columns:
    df_clean = df[["Question", "Answer"]].rename(columns={"Question": "question", "Answer": "answer"}).dropna()
else:

    df_clean = df.iloc[:, :2]
    df_clean.columns = ["question", "answer"]
    df_clean = df_clean.dropna()


output_filename = "medquad.csv"
df_clean.to_csv(output_filename, index=False)

print(f"Dataset successfully created and saved locally as '{output_filename}'!")
print(f"Total Question-Answer pairs loaded: {len(df_clean)}")
print("\nPreview of the dataset:")
print(df_clean.head(3))

Dataset successfully created and saved locally as 'medquad.csv'!
Total Question-Answer pairs loaded: 16407

Preview of the dataset:
                                            question  \
0       What is (are) keratoderma with woolly hair ?   
1  How many people are affected by keratoderma wi...   
2  What are the genetic changes related to kerato...   

                                              answer  
0  Keratoderma with woolly hair is a group of rel...  
1  Keratoderma with woolly hair is rare; its prev...  
2  Mutations in the JUP, DSP, DSC2, and KANK2 gen...  


In [1]:
%%writefile README.md
# Medical Q&A Chatbot (MedQuAD)

A specialized medical question-answering chatbot built using the MedQuAD dataset, featuring a TF-IDF retrieval mechanism, basic medical entity recognition (MER), and an interactive Streamlit user interface designed to run seamlessly on Google Colab.

## Features
- **Knowledge Retrieval:** Uses TF-IDF vectorization and cosine similarity to match user queries with verified medical Q&A pairs.
- **Medical Entity Recognition (MER):** Extracts core health entities (diseases, symptoms, and treatments) from text via pattern and keyword matching.
- **Interactive Web UI:** Built with Streamlit for a clean, user-friendly frontend experience.
- **Cloud-Optimized:** Designed to execute efficiently within Google Colab using lightweight data frames.

## Tech Stack
- **Python**
- **Pandas** (Data handling & preprocessing)
- **Scikit-Learn** (TF-IDF vectorizer & cosine similarity metrics)
- **Streamlit** (Web app framework)
- **LocalTunnel / Pyngrok** (Colab port tunneling)

## Disclaimer
This project is built strictly for educational and internship demonstration purposes. It does not constitute professional medical advice, diagnosis, or treatment.

Writing README.md
